In [139]:
from torch_geometric.data import HeteroData
import torch
import pickle
import numpy as np
import torch

In [140]:
with open("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\picklefiles\\final_eal.pkl", 'rb') as eal:
    data = pickle.load(eal)

In [141]:
ent = [data[i][0] for i in range(len(data))]

In [142]:
asp = [data[i][1] for i in range(len(data))]

In [143]:
count = 0
for e in ent:
    count += len(e['entities'])

In [144]:
count

467

In [145]:
graph = HeteroData()

In [146]:
len(ent)

100

In [147]:
def get_node_id(data, subject = "a_entities"):
    res = []
    dic = {}
    if subject == "target_entity":
        for item in data:
            target, _ = item
            res.append(target['id'])
        return list(map(int,res))
    elif subject == "t_entities":
        i = 0
        for item in data:
            target,_ = item
            for ent in target['entities']:
                dic[ent['eid']] = i
                i += 1
        return dic
    elif subject == "aspect_entity":
        i = 0
        for item in data:
            _, target = item
            dic[target["id"]] = i
            i+=1
            for l in target["candidate_aspects"]:
                if l["aspect_name"] != target["true_aspect"]:
                    dic[l["id"]] = i
                    i+=1           
        return dic
    elif subject == 'a_entities':
        i = 0
        for item in data:
            _, target = item
            for l in target["candidate_aspects"]:
                for el in l["entities"]:
                    dic[el['eid']] = i
                    i+= 1
        
        return dic
        
    

In [148]:
t_key = get_node_id(data, subject = "t_entities")
tid = list(get_node_id(data, subject = "t_entities").values())

In [149]:
def get_edge_entities(data, key , type = 'target_entity', **kwargs):
    edge = []
    if type == 'target_entity':
        for item in data:
            ent, _ = item
            ent_id = int(ent['id'])
            for item in ent['entities']:
                edge.append([ent_id, key[item['eid']]])
        return np.array(edge).T

    elif type =='aspect_entity':
        for item in data:
            _, asp = item
            asp_id = asp['id']
            #print(asp_id)
            for cand in asp['candidate_aspects']:
                if cand['aspect_name'] == asp['true_aspect']:
                    for e in cand['entities']:
                        edge.append([aspdict[asp_id], key[e['eid']]])
        return np.array(edge).T
                    
    


In [150]:
def get_target_edges(data, asp_dict):
    edge = []
    for item in data:
        ent, asp = item
        tid = int(ent['id'])
        aspid = asp_dict[asp['id']]
        edge.append([tid, aspid])
    return np.array(edge).T

In [151]:
ent_edge_index = torch.tensor(get_edge_entities(data, t_key))

In [152]:
aspdict = get_node_id(data,subject = "aspect_entity")
asp_id = list(aspdict.values())

In [153]:
a_entdict = get_node_id(data, subject = "a_entities")
a_ent_id = list(a_entdict.values())


In [154]:
len(list(a_entdict.keys()))

18252

In [155]:
asp_edge_index = torch.tensor(get_edge_entities(data, key = a_entdict, type = 'aspect_entity', a_entdict = a_entdict))

In [156]:
target_edge_index = torch.tensor(get_target_edges(data, aspdict))

In [157]:
graph['target_entity'].num_nodes = len(ent)
graph['target_entity'].node_id = get_node_id(data, subject = 'target_entity')
graph['t_entities'].num_nodes = count
graph['t_entities'].node_id = tid
graph['target_entity', 'associated_with', 't_entities'] = ent_edge_index

graph["aspect_entity"].num_nodes = len(aspdict)
graph["aspect_entity"].node_id = asp_id
graph['a_entities'].num_nodes = len(a_ent_id)
graph['a_entities'].node_id = a_ent_id
graph['aspect_entity', 'associated_with', 'a_entities'] = asp_edge_index

graph['target_entity', 'linked_to', 'aspect_entity'] = target_edge_index


In [158]:
graph

HeteroData(
  (target_entity, associated_with, t_entities)=[2, 467],
  (aspect_entity, associated_with, a_entities)=[2, 8922],
  (target_entity, linked_to, aspect_entity)=[2, 100],
  target_entity={
    num_nodes=100,
    node_id=[100]
  },
  t_entities={
    num_nodes=467,
    node_id=[467]
  },
  aspect_entity={
    num_nodes=640,
    node_id=[640]
  },
  a_entities={
    num_nodes=18252,
    node_id=[18252]
  }
)